<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>Generative AI and Anthropic</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
from collections import defaultdict

import numpy as np

import anthropic
import os
import base64
import httpx
from IPython.display import display, Markdown, Image

import matplotlib.pyplot as plt

import nltk
from nltk.corpus import reuters
from nltk import bigrams, trigrams

import tqdm
from tqdm.notebook import tqdm

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 87485bac20625f43da5809d38ec119aca4321f71

tqdm      : 4.67.1
IPython   : 9.8.0
anthropic : 0.75.0
matplotlib: 3.10.7
httpx     : 0.28.1
numpy     : 2.3.5
watermark : 2.5.0
nltk      : 3.9.2



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# "Small" Language Model

In [4]:
model = defaultdict(lambda: defaultdict(lambda: 0))

We start by counting number of trigram co-occurrences

In [5]:
for sentence in tqdm(reuters.sents(), total=54_716):
    for w1, w2, w3 in trigrams(sentence, pad_right=True, pad_left=True):
        bigram = (w1, w2)
        model[bigram][w3] += 1

  0%|          | 0/54716 [00:00<?, ?it/s]

And normalizing the probabilities for each bigram. 

In [6]:
for bigram in model:
    total_count = float(sum(model[bigram].values()))

    for w3 in model[bigram]:
        model[bigram][w3] /= total_count

Our language model is just a weighted mapping between each bigram and the possible next words.model[("the", "United")]

In [7]:
model[("the", "United")]

defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>,
            {'States': 0.880672268907563,
             'Kingdom': 0.011764705882352941,
             'Arab': 0.052100840336134456,
             'Permanent': 0.0016806722689075631,
             'Steelworkers': 0.0033613445378151263,
             'Nations': 0.025210084033613446,
             'Coconut': 0.0067226890756302525,
             'State': 0.0033613445378151263,
             'Democratic': 0.0016806722689075631,
             'Food': 0.008403361344537815,
             'Automobile': 0.0016806722689075631,
             'acquisition': 0.0016806722689075631,
             'Rubber': 0.0016806722689075631})

In [8]:
model[("United", "Kingdom")]

defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>,
            {',': 0.21428571428571427,
             'and': 0.21428571428571427,
             'blender': 0.07142857142857142,
             ')': 0.14285714285714285,
             'company': 0.07142857142857142,
             'operations': 0.07142857142857142,
             'assets': 0.07142857142857142,
             'Ltd': 0.07142857142857142,
             '.': 0.07142857142857142})

<center>
     <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>